<a href="https://colab.research.google.com/github/shinigami-rohit/atschecker/blob/main/Copy_of_jobrecommendation_and_atschecker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas scikit-learn PyPDF2 nltk

In [ ]:
import pandas as pd

jobs = pd.DataFrame({
    "job_title": [
        "Data Scientist", "ML Engineer", "AI Engineer",
        "Web Developer", "Frontend Developer",
        "Backend Developer", "Android Developer",
        "Data Analyst", "Software Engineer"
    ],
    "skills": [
        "python machine learning pandas numpy",
        "python sklearn tensorflow pytorch",
        "deep learning nlp python",
        "html css javascript",
        "react js ui ux",
        "node js django flask",
        "java kotlin android",
        "excel sql python",
        "c++ java python"
    ],
    "description": [
        "analyze data build models",
        "train machine learning systems",
        "build AI systems NLP",
        "build websites",
        "design user interface",
        "develop backend systems",
        "build mobile apps",
        "analyze business data",
        "software development"
    ]
})

jobs["text"] = jobs["skills"] + " " + jobs["description"]

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

jobs["text"] = jobs["text"].apply(clean_text)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(jobs["text"])
y = jobs["job_title"]

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("Model Accuracy:", model.score(X_test, y_test))

Model Accuracy: 0.0


In [ ]:
from google.colab import files
import PyPDF2

uploaded = files.upload()

def extract_text(file_name):
    text = ""
    with open(file_name, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text()
    return text

Saving Ritik_Raj_Resume.pdf to Ritik_Raj_Resume.pdf


In [ ]:
skills_list = [
    "python", "java", "c++", "machine learning",
    "deep learning", "html", "css", "javascript",
    "react", "node", "sql", "excel", "django", "flask"
]

def extract_skills(text):
    found_skills = []
    text = text.lower()
    for skill in skills_list:
        if skill in text:
            found_skills.append(skill)
    return found_skills

In [ ]:
def calculate_ats(resume_text, job_text):
    resume_words = set(resume_text.split())
    job_words = set(job_text.split())

    match = resume_words.intersection(job_words)
    score = len(match) / len(job_words)

    return round(score * 100, 2)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def full_analysis(resume_text):
    resume_clean = clean_text(resume_text)

    # Skill extraction
    skills = extract_skills(resume_clean)

    # Vector transform
    resume_vector = vectorizer.transform([resume_clean])

    # Classification prediction
    predicted_job = model.predict(resume_vector)[0]

    # Similarity recommendation
    similarity = cosine_similarity(resume_vector, X)
    top_indices = similarity[0].argsort()[-3:][::-1]

    recommended_jobs = jobs.iloc[top_indices]

    # ATS score for best match
    best_job_text = recommended_jobs.iloc[0]["text"]
    ats_score = calculate_ats(resume_clean, best_job_text)

    return {
        "Predicted Job": predicted_job,
        "Skills Found": skills,
        "ATS Score": ats_score,
        "Top Recommendations": recommended_jobs["job_title"].tolist()
    }

In [ ]:
file_name = list(uploaded.keys())[0]
resume_text = extract_text(file_name)

result = full_analysis(resume_text)

print("\n🎯 FINAL RESULT\n")
for key, value in result.items():
    print(f"{key}: {value}")


🎯 FINAL RESULT

Predicted Job: Data Scientist
Skills Found: ['excel']
ATS Score: 33.33
Top Recommendations: ['Data Analyst', 'Data Scientist', 'AI Engineer']
